# INSERT Algorithm Test — Visualización

Muestra el resultado del test de integración del algoritmo INSERT.
Los servicios se dividen 50/50 desde una solución OFFLINE de referencia:

- **Azul** — labores preassigned (congeladas, provenientes del OFFLINE)
- **Verde** — labores insertadas por el algoritmo INSERT
- **Naranja** — movimiento del conductor entre servicios
- **Gris** — tiempo libre

> Este patrón de visualización (`origin = preassigned | inserted`) aplica también para BUFFER_REACT.

In [1]:
from __future__ import annotations

import json
import sys
from collections import defaultdict
from datetime import timedelta
from pathlib import Path
from typing import Dict, List, Optional
from zoneinfo import ZoneInfo

import pandas as pd
import plotly.graph_objects as go

_PROJECT_ROOT = Path("..").resolve()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

In [2]:
# ── Configuration ────────────────────────────────────────────────────────────
# Set RESULTS_DIR to the specific run dir, or leave None to auto-pick the latest.

RESULTS_DIR: Optional[Path] = None

# _base = _PROJECT_ROOT / "experiments" / "insert_test" / "results"
_base = _PROJECT_ROOT / "experiments" / "phase3" / "insert" / "output"
if RESULTS_DIR is None:
    _runs = sorted(_base.glob("run-insert-*"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not _runs:
        raise FileNotFoundError(
            f"No runs found under {_base}.\n"
            "Run experiments/insert_test/run_test.py first."
        )
    RESULTS_DIR = _runs[0]

print(f"Loading from: {RESULTS_DIR}")

FileNotFoundError: No runs found under /Users/jbeta/Documents/AlfredEnvs/AlfredPROD/experiments/phase3/insert/output.
Run experiments/insert_test/run_test.py first.

In [3]:
# ── Load results ─────────────────────────────────────────────────────────────
results = pd.read_csv(RESULTS_DIR / "insert_results.csv", low_memory=False)
moves   = pd.read_csv(RESULTS_DIR / "insert_moves.csv",   low_memory=False)
summary = json.loads((RESULTS_DIR / "test_summary.json").read_text())

# Use .apply(pd.Timestamp) — vectorized to_datetime silently drops rows
# with mixed sub-second timezone offsets (e.g. "07:09:23.924691-05:00").
for _df in (results, moves):
    for _col in ("actual_start", "actual_end"):
        if _col in _df.columns:
            _df[_col] = _df[_col].apply(lambda x: pd.Timestamp(x) if pd.notna(x) else pd.NaT)
if "schedule_date" in results.columns:
    results["schedule_date"] = results["schedule_date"].apply(
        lambda x: pd.Timestamp(x) if pd.notna(x) else pd.NaT
    )

print(f"results : {len(results)} rows")
print(f"moves   : {len(moves)} rows  |  nulls actual_start: {moves['actual_start'].isna().sum()}")
print(f"seed    : {summary['config']['seed']}")
print(f"distance: {summary['config']['distance_method']} | time: {summary['config']['time_method']}")

results : 41 rows
moves   : 117 rows  |  nulls actual_start: 0
seed    : 42
distance: haversine | time: speed_based


---
## Resumen

In [5]:
# ── Summary stats ─────────────────────────────────────────────────────────────
_r = summary.get("results", {})
_s = summary.get("split", {})

_pre  = results[results["origin"] == "preassigned"]
_ins  = results[results["origin"] == "inserted"]

_ins_assigned   = _ins["assigned_driver"].notna().sum() if "assigned_driver" in _ins.columns else 0
_ins_unassigned = _ins["assigned_driver"].isna().sum()  if "assigned_driver" in _ins.columns else 0
_ins_infeasible = int(_ins["is_infeasible"].sum())       if "is_infeasible"  in _ins.columns else 0
_drivers_used   = results["assigned_driver"].dropna().nunique()

pd.DataFrame([
    {"metric": "total_services_baseline",  "value": _r.get("baseline_services")},
    {"metric": "services_preassigned",     "value": _s.get("preassigned_count")},
    {"metric": "services_to_insert",       "value": _s.get("insert_count")},
    {"metric": "labors_preassigned",       "value": len(_pre)},
    {"metric": "labors_insert_input",      "value": len(_ins)},
    {"metric": "labors_insert_assigned",   "value": int(_ins_assigned)},
    {"metric": "labors_insert_unassigned", "value": int(_ins_unassigned)},
    {"metric": "labors_infeasible",        "value": _ins_infeasible},
    {"metric": "drivers_used_total",       "value": int(_drivers_used)},
]).set_index("metric")

,value
metric,
total_services_baseline,35
services_preassigned,17
services_to_insert,18
labors_preassigned,19
labors_insert_input,22
labors_insert_assigned,20
labors_insert_unassigned,2
labors_infeasible,0
drivers_used_total,18


---
## Gantt — Timelines por Conductor

| Color | Significado |
|-------|-------------|
| 🔵 Azul | Labor preassigned |
| 🟠 Naranja | Movimiento — preassigned |
| 🟢 Verde | Labor insertada (INSERT) |
| 🫒 Verde oliva | Movimiento — insertado |
| ⬜ Gris | Tiempo libre |

In [6]:
import ipywidgets as widgets
from IPython.display import display

# ── Colors & labels ───────────────────────────────────────────────────────────
_COLORS = {
    "FREE_TIME":              "rgba(190, 190, 190, 0.35)",
    "preassigned_move":       "rgba(250, 155,  45, 0.85)",  # orange
    "inserted_move":          "rgba(120, 180,  80, 0.75)",  # olive-green
    "preassigned_labor":      "rgba( 55, 115, 200, 0.88)",  # blue
    "inserted_labor":         "rgba( 40, 170,  80, 0.90)",  # green
}
_LABELS = {
    "FREE_TIME":              "Free time",
    "preassigned_move":       "Driver move (preassigned)",
    "inserted_move":          "Driver move (inserted)",
    "preassigned_labor":      "Labor preassigned",
    "inserted_labor":         "Labor inserted",
}
_ORDER = ["FREE_TIME", "preassigned_move", "inserted_move", "preassigned_labor", "inserted_labor"]
_MIN_VT_MS = 5 * 60 * 1000
_BOGOTA_TZ  = "America/Bogota"


def _to_plotly_dt(ts) -> str:
    if pd.isna(ts):
        return ""
    ts = pd.Timestamp(ts)
    if ts.tzinfo is not None:
        ts = ts.tz_convert(_BOGOTA_TZ).tz_localize(None)
    return ts.isoformat()


def _classify(row) -> str:
    name   = str(row.get("labor_name", "") or "").strip().upper()
    origin = str(row.get("origin",     "") or "").strip().lower()
    if name == "FREE_TIME":
        return "FREE_TIME"
    if name == "DRIVER_MOVE":
        return "inserted_move" if origin == "inserted" else "preassigned_move"
    return "inserted_labor" if origin == "inserted" else "preassigned_labor"


def _build_traces(df: pd.DataFrame, show_legend: bool = True) -> list:
    by_type = defaultdict(list)
    for _, row in df.iterrows():
        by_type[_classify(row)].append(row)

    traces = []
    for seg_type in _ORDER:
        segs = by_type.get(seg_type, [])
        if not segs:
            continue
        base_vals, x_vals, y_vals, hovers = [], [], [], []
        for row in segs:
            start, end = row["actual_start"], row["actual_end"]
            if pd.isna(start) or pd.isna(end):
                continue
            dur_ms = (end - start).total_seconds() * 1000
            if dur_ms < 0:
                continue
            is_labor = seg_type in ("preassigned_labor", "inserted_labor")
            if is_labor and dur_ms == 0:
                dur_ms = _MIN_VT_MS
            elif not is_labor and dur_ms == 0:
                continue

            base_vals.append(_to_plotly_dt(start))
            x_vals.append(dur_ms)
            y_vals.append(str(row["assigned_driver"]))

            dur_min = dur_ms / 1_000 / 60
            ht = (
                f"<b>{_LABELS[seg_type]}</b><br>"
                f"Driver: {row['assigned_driver']}<br>"
                f"Labor: {row.get('labor_id', '?')}<br>"
                f"Service: {row.get('service_id', '?')}<br>"
                f"Start: {pd.Timestamp(start).strftime('%H:%M')}<br>"
                f"End:   {pd.Timestamp(end).strftime('%H:%M')}<br>"
                f"Duration: {dur_min:.1f} min"
            )
            dist = row.get("distance_km", 0) or 0
            if dist > 0:
                ht += f"<br>Distance: {dist:.2f} km"
            hovers.append(ht)

        if not x_vals:
            continue
        traces.append(go.Bar(
            x=x_vals, y=y_vals, base=base_vals, orientation="h",
            name=_LABELS[seg_type],
            marker_color=_COLORS[seg_type],
            hovertext=hovers, hoverinfo="text",
            showlegend=show_legend,
            legendgroup=seg_type,
        ))
    return traces


def build_insert_gantt(
    df: pd.DataFrame,
    label: str = "INSERT Test",
    show_legend: bool = True,
) -> go.Figure:
    df = df.dropna(subset=["assigned_driver", "actual_start", "actual_end"]).copy()
    if df.empty:
        return go.Figure().update_layout(title_text=f"{label} — no data")

    drivers_ordered = (
        df.groupby("assigned_driver")["actual_start"]
        .min().sort_values().index.tolist()
    )
    fig = go.Figure(data=_build_traces(df, show_legend=show_legend))

    pad = pd.Timedelta(minutes=30)
    fig.update_layout(xaxis=dict(
        type="date", tickformat="%H:%M", title_text="Time (Bogotá)",
        range=[
            _to_plotly_dt(df["actual_start"].min() - pad),
            _to_plotly_dt(df["actual_end"].max() + pad),
        ],
    ))
    fig.update_yaxes(
        categoryorder="array",
        categoryarray=list(reversed(drivers_ordered)),
        title_text="Driver",
    )
    row_h = max(35, 600 // max(len(drivers_ordered), 1))
    fig.update_layout(
        title_text=label,
        barmode="overlay",
        height=max(420, row_h * len(drivers_ordered) + 160),
        legend=dict(orientation="h", yanchor="bottom", y=1.06, xanchor="center", x=0.5),
        hovermode="closest",
        margin=dict(l=120, r=20, t=100, b=60),
    )
    return fig


_df_check = moves.dropna(subset=["assigned_driver", "actual_start", "actual_end"])
print(f"moves with valid timestamps: {len(_df_check)} / {len(moves)}")

moves with valid timestamps: 117 / 117


In [7]:
# ── Gantt 1: preassigned schedule only ───────────────────────────────────────
_moves_pre = moves[moves["origin"] == "preassigned"]
build_insert_gantt(_moves_pre, label=f"Preassigned schedule — {RESULTS_DIR.name}").show()

In [8]:
# ── Gantt 2: full schedule — preassigned + inserted ───────────────────────────
build_insert_gantt(moves, label=f"Full schedule (preassigned + inserted) — {RESULTS_DIR.name}").show()

---
## Gantt por conductor — detalle

Selecciona un conductor para ver su timeline completo (preassigned + inserted).

In [11]:
# ── Per-driver Gantt with dropdown ───────────────────────────────────────────
_all_drivers = sorted(
    moves["assigned_driver"].dropna().unique().tolist(),
    key=lambda d: str(d),
)

_drv_dropdown = widgets.Dropdown(
    options=_all_drivers,
    description="Driver:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px"),
)
_out = widgets.Output()


def _render_driver(driver_id):
    _out.clear_output(wait=True)
    with _out:
        df_drv = moves[moves["assigned_driver"] == driver_id]
        build_insert_gantt(df_drv, label=f"Driver {driver_id}").show()


_drv_dropdown.observe(lambda change: _render_driver(change["new"]), names="value")
_render_driver(_drv_dropdown.value)
display(_drv_dropdown, _out)

Dropdown(description='Driver:', layout=Layout(width='300px'), options=('Alberto Mora', 'Ancisar Echavarría', '…

Output()

---
## Labores insertadas — detalle

In [ ]:
_inserted = results[results["origin"] == "inserted"].copy()

_display_cols = [c for c in [
    "service_id", "labor_id", "labor_type", "assigned_driver",
    "actual_start", "actual_end", "is_infeasible", "infeasibility_cause_code",
] if c in _inserted.columns]

_inserted[_display_cols].sort_values(["assigned_driver", "actual_start"]).reset_index(drop=True)